# RAG With Knowledge Graph using Neo4j

This notebook builds a Retrieval Augmented Generation system that combines a knowledge graph stored in Neo4j with a vector index. The system reads text, extracts entities and relationships with a language model, stores those entities and relationships as a graph, and also stores the text as embeddings for semantic search. When a user asks a question, the system pulls relevant facts from both the graph and the vector index, combines them, and passes that combined context to a language model to produce the final answer.

The notebook is organized below into labeled categories. Each category has a short explanation describing what the code inside it does and why it matters for the overall pipeline. The original code itself is kept exactly as it was written.

## About the LangChain Packages Used

**langchain core**
Contains simple, core abstractions that have emerged as a standard, along with the LangChain Expression Language used to compose these components together.

**langchain community**
Contains third party integrations, including the Neo4j graph and vector store integrations used throughout this notebook.

**langchain**
Contains higher level and use case specific chains, agents, and retrieval algorithms that form the cognitive architecture of the application.

## 1. Installing Required Libraries

Before anything else, the environment needs the packages that power this notebook. `langchain`, `langchain community`, and `langchain openai` provide the chains, integrations, and OpenAI model wrappers. `langchain experimental` provides the `LLMGraphTransformer`, which is the component that turns plain text into graph structured data. `neo4j` is the official Python driver used to talk to the Neo4j database. `wikipedia` lets the notebook pull source articles directly from Wikipedia. `tiktoken` is used internally for token counting when splitting text. `yfiles_jupyter_graphs` renders an interactive visual of the graph directly inside the notebook.

This step only needs to run once per environment.

In [ ]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs

## 2. Importing Required Libraries

This category gathers every import used later in the notebook. Grouping them together makes it clear, at a glance, which building blocks the pipeline depends on:

1. `RunnableBranch`, `RunnableLambda`, `RunnableParallel`, and `RunnablePassthrough` from `langchain_core.runnables` are the composition primitives used to build the final conversational chain.
2. `ChatPromptTemplate` and `PromptTemplate` are used to construct the prompts sent to the language model.
3. `Tuple`, `List`, and `Optional` from the standard `typing` module are used for type hints, which make the helper functions easier to read and safer to maintain.
4. `AIMessage`, `HumanMessage`, and `StrOutputParser` handle chat message formatting and parsing of the model output into plain text.
5. `ConfigurableField` allows runnable components to expose configurable parameters.
6. `GraphWidget` from `yfiles_jupyter_graphs` and `GraphDatabase` from `neo4j` are used later to visualize the graph interactively.
7. The `os` module is used to store credentials as environment variables so that downstream LangChain integrations can pick them up automatically.

In [ ]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [ ]:
from typing import Tuple, List, Optional

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from langchain_core.runnables import ConfigurableField

In [ ]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase


In [ ]:
import os

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [ ]:
from langchain_community.vectorstores import Neo4jVector

## 3. Setting Up API Keys and Environment Variables

Every external service used in this notebook needs a credential. The OpenAI API key authorizes calls to the language model and the embedding model. The Neo4j URI, username, and password authorize the connection to the graph database instance that will store the extracted knowledge graph.

The credentials are first retrieved from Colab secret storage using `userdata.get`, then written into `os.environ` so that LangChain integrations such as `Neo4jGraph` and `ChatOpenAI` can read them automatically without needing to pass them explicitly into every function call. Keeping secrets in environment variables rather than hard coded inline is a common pattern for keeping notebooks reusable and shareable.

In [ ]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [ ]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [ ]:
NEO4J_URI="neo4j+s://7b0ac3fd.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="al6q_y6NWn8e98YXHElSBED010quYdte4FaNxL-hESg"

In [ ]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

## 4. Connecting to the Neo4j Graph Database

With the credentials available as environment variables, `Neo4jGraph` can open a connection to the database instance automatically. This `graph` object is the central handle used throughout the rest of the notebook for writing graph data, running Cypher queries, and creating indexes.

In [ ]:
from langchain_community.graphs import Neo4jGraph

In [ ]:
graph = Neo4jGraph()

## 5. Loading Source Data from Wikipedia

The knowledge graph needs raw text to learn from. `WikipediaLoader` fetches articles related to the query "Elizabeth I" directly from Wikipedia and returns them as LangChain `Document` objects. Printing the length and previewing the first few documents confirms that the data was retrieved successfully before any further processing happens.

In [ ]:
from langchain.document_loaders import WikipediaLoader
raw_documents = WikipediaLoader(query="Elizabeth I").load()

In [ ]:
len(raw_documents)

In [ ]:
raw_documents[:3]

## 6. Splitting Documents into Chunks

Language models have a limited context window, and graph extraction quality tends to degrade on very long passages. `TokenTextSplitter` breaks the raw documents into smaller chunks of five hundred and twelve tokens each, with a twenty four token overlap between consecutive chunks so that context is not lost at chunk boundaries. Only the first three raw documents are used here to keep the example fast and inexpensive to run.

In [ ]:
from langchain.text_splitter import TokenTextSplitter
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])

## 7. Initializing the Language Model

A single `ChatOpenAI` instance, using the `gpt 3.5 turbo` model with temperature set to zero, is created here and reused for every downstream task that needs a language model, including graph extraction, entity extraction, and final answer generation. A temperature of zero makes the model deterministic, which is desirable for structured extraction tasks where consistency matters more than creativity.

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-0125")

## 8. Transforming Text into Graph Documents

This is the step where unstructured text becomes structured knowledge. `LLMGraphTransformer` uses the language model defined above to read each chunk of text and identify entities such as people, places, and organizations, along with the relationships that connect them. The result, `graph_documents`, is a list of graph structured representations, one for each input chunk, containing nodes and relationships that can later be written into Neo4j. Printing `graph_documents` lets you inspect exactly what the model extracted before committing it to the database.

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(llm=llm)

In [ ]:
graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [ ]:
graph_documents

## 9. Storing the Graph Documents in Neo4j

The extracted nodes and relationships are written into the Neo4j database using `add_graph_documents`. Setting `baseEntityLabel=True` gives every extracted node a shared base label, which makes it easier to query across different entity types later. Setting `include_source=True` also creates a link back to the original source document for each entity, which preserves traceability between the graph and the text it came from.

In [ ]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

## 10. Visualizing the Knowledge Graph

Seeing the graph visually is a useful sanity check before building retrieval on top of it. This category defines a Cypher query that matches nodes and relationships while excluding the internal `MENTIONS` relationship, then uses a small helper function, `showGraph`, to open a direct driver session, run the query, and render the results using an interactive `GraphWidget`. Running `showGraph()` displays the graph inline in the notebook, with node labels mapped to each entity's `id` property for readability.

In [ ]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

In [ ]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [ ]:
def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    display(widget)
    return widget

In [ ]:
showGraph()

## 11. Building a Vector Index for Semantic Search

A knowledge graph is excellent for precise, structured facts, but it does not capture the full nuance of the original text. To cover that gap, this category builds a vector index directly on top of the `Document` nodes already stored in Neo4j from the earlier `include_source=True` step. `Neo4jVector.from_existing_graph` embeds the `text` property of each `Document` node using `OpenAIEmbeddings` and stores the resulting vector in an `embedding` property. Setting `search_type="hybrid"` allows the index to combine vector similarity search with keyword based fulltext search, which tends to produce more robust retrieval than either method alone.

In [ ]:
from typing import Tuple, List, Optional

In [ ]:
from langchain_community.vectorstores import Neo4jVector

In [ ]:
from langchain_openai import OpenAIEmbeddings
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

## 12. Creating a Fulltext Index for Entity Search

To look up entities in the graph by name quickly and with tolerance for spelling variation, a fulltext index named `entity` is created on the `id` property of every `__Entity__` node. This index powers the structured retrieval step later, where entity names extracted from a user question are matched against nodes already stored in the graph.

In [ ]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

## 13. Extracting Entities from User Questions

Before the graph can be queried for a user's question, the system needs to know which entities the question is actually about. This category defines an `Entities` schema using Pydantic, describing a list of person, organization, or business names. The `entity_chain` combines a prompt instructing the model to act as an entity extractor with `llm.with_structured_output(Entities)`, which forces the model's response to conform to that schema. The example call shows the chain correctly pulling `"Amelia Earhart"` out of a sample question.

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [ ]:
entity_chain = prompt | llm.with_structured_output(Entities)

In [ ]:
entity_chain.invoke({"question": "Where was Amelia Earhart born?"}).names

## 14. Structured Retrieval Using the Graph

This category performs the actual graph lookup for a question. `remove_lucene_chars` strips characters that would otherwise break a Lucene fulltext query. `generate_full_text_query` turns the extracted entity name into a fuzzy Lucene query, appending a `~2` edit distance tolerance to each word so that minor spelling differences still match. `structured_retriever` ties it together: it calls `entity_chain` to pull entity names out of the question, then for each entity it queries the `entity` fulltext index, finds the matching node, and walks both outgoing and incoming relationships (excluding `MENTIONS`) up to fifty results, formatting each fact as `source relationship target`. The final print statement demonstrates the function pulling structured facts about Elizabeth I directly out of the graph.

In [ ]:
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

In [ ]:
def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


In [ ]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [ ]:
print(structured_retriever("Who is Elizabeth I?"))

## 15. Combining Structured and Unstructured Retrieval

Neither retrieval method alone tells the whole story, so this category merges them. The `retriever` function calls `structured_retriever` to gather graph facts and also calls `vector_index.similarity_search` to gather the most semantically relevant text chunks. Both results are concatenated into a single formatted string labeled `Structured data` and `Unstructured data`, which becomes the context passed to the language model when answering the user's question. This hybrid context is the core idea behind graph augmented retrieval augmented generation.

In [ ]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

## 16. Handling Follow up Questions with Chat History

Real conversations often include follow up questions that only make sense with prior context, such as "When was she born?" after previously discussing Elizabeth I. This category defines a condensing prompt, `CONDENSE_QUESTION_PROMPT`, that rewrites a follow up question into a standalone question using the chat history. `_format_chat_history` converts the raw list of human and AI text pairs into proper `HumanMessage` and `AIMessage` objects. Finally, `_search_query` is a `RunnableBranch` that checks whether chat history was actually provided. If it was, the branch condenses the question using the prompt, the language model, and a string output parser. If not, it simply passes the original question straight through unchanged.

In [ ]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [ ]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [ ]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [ ]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

## 17. Building the Final RAG Chain

This category assembles every previous piece into one runnable pipeline. The `template` instructs the model to answer strictly using the provided context, in natural and concise language, which helps reduce hallucination. `RunnableParallel` runs two branches at once: the `context` branch sends the (possibly condensed) question through `_search_query` and then through the combined `retriever` to gather graph and vector facts, while the `question` branch simply passes the original input through unchanged. Both outputs feed into the `prompt`, then into the `llm`, and finally through `StrOutputParser` to produce a clean text answer. The result, `chain`, is the complete conversational, graph aware retrieval augmented generation pipeline.

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

## 18. Testing the RAG Pipeline

Finally, the complete chain is exercised with two calls. The first call asks a standalone question with no prior history, confirming that the graph and vector retrieval, combined with the language model, produce a correct grounded answer. The second call asks a follow up question, "When was she born?", along with a `chat_history` entry from the earlier turn. This proves that the condensing step correctly rewrites the vague follow up into a standalone question such as "When was Elizabeth I born?" before retrieval happens, so the pipeline can still answer correctly even though the follow up question alone does not mention Elizabeth I by name.

In [ ]:
chain.invoke({"question": "Which house did Elizabeth I belong to?"})

In [ ]:
chain.invoke(
    {
        "question": "When was she born?",
        "chat_history": [("Which house did Elizabeth I belong to?", "House Of Tudor")],
    }
)

## Conclusion

This notebook demonstrated an end to end pipeline for building a retrieval augmented generation system that is grounded in a knowledge graph rather than plain text chunks alone. Starting from raw Wikipedia articles about Elizabeth I, the pipeline split the text, used a language model to extract entities and relationships, and stored that structured knowledge inside a Neo4j graph database. On top of that same data, a hybrid vector index was built so that semantically relevant passages could still be retrieved even when a fact was not explicitly captured as a graph relationship.

At query time, the system extracts entities from the user's question, traverses the graph around those entities to gather precise structured facts, and simultaneously performs a similarity search to gather supporting unstructured context. These two sources are merged and handed to the language model, which produces a concise, grounded answer. A conversational layer on top of this, built with a question condensing step, allows the system to correctly interpret follow up questions that depend on earlier turns in the conversation.

The overall pattern shown here, combining a knowledge graph with vector search rather than relying on either one alone, tends to produce answers that are both factually precise and contextually rich. This makes it a strong foundation for building question answering systems over structured domains such as biographies, organizational data, product catalogs, or any dataset where the relationships between entities carry as much meaning as the text itself.